In [1]:
import json
import os
from os import listdir
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import torch
from interactive_seg_backend import (
    FeatureConfig,
    TrainingConfig,
    concat_feats,
    featurise,
    train_and_apply,
)
from interactive_seg_backend.file_handling import load_labels
from interactive_seg_backend.utils import class_avg_miou
from matplotlib.lines import Line2D
from PIL import Image
from skimage.color import label2rgb
from skimage.transform import resize

from dinosaw.linear_probe import (
    LinearProbeResult,
    RampTypes,
    do_linear_probe,
)
from dinosaw.utils import (
    COLORS,
    add_custom_font,
    apply_labels_as_overlay,
    do_2D_pca,
    get_features,
    hide_axis,
)
from dinosaw.wrappers import MODEL_NAMES, ModelTypes
from dinosaw.wrappers.register_models import get_model_

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda:0'
half = False

type EncoderKey = ModelTypes | Literal['classical']

SCORES_CACHE_PATH = "saved/s16_scores.json"
LOAD_RESULTS: bool = True
INCLUDE_CLASSICAL_BASELINE: bool = False
USE_MICRO_DS_R2: bool = True
PLOT_BOTH_CLASSICAL_MODES: bool = True
ADD_CLASSICAL_FEATS: bool = False

2026-11-09 22:22:14 | I | multiscale_classical_cpu.py:  46 | N CPUS: 10
2026-11-09 22:22:14 | W | sam_onnx.py                :  19 | ONNX SAM unavailable!


/home/ronan/Documents/academic/dino-saw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
vit_models: tuple[ModelTypes, ...] = (
    'dino_s',
    'dinov2_s',
    'alibi_coco_dinov2_s',
    'sam_b',
    'clip_b',
    'mae_b',
    'in1k_b',
)
all_encoders: tuple[EncoderKey, ...] = (*vit_models, 'classical')

f_cfg = FeatureConfig()

tr_cfg = TrainingConfig(
    feature_config=f_cfg,
    CRF=True,
    classifier='xgb',
    CRF_params={
        "label_confidence": 0.6,
        "sxy_g": [1, 1],
        "sxy_b": [30, 30],
        "s_rgb": [13, 13, 13],
        "compat_g": 10,
        "compat_b": 10,
        "n_infer": 10,
    },
    classifier_params={"class_weight": "balanced", "max_depth": 8},
)

In [3]:
ds_folder = 'data/trainable_seg'
side_lengths = {
    'au_foil': 737,
    'anode': 756,
    'curtains': 756,
    'rome': 756,
    'nmc_wide': 808,
    'cracks': 602,
    'butterflies': 756,
    'bimodal': 839,
    'NMC': 602,
    'steel': 602,
    'pre_kint': 1024,
}

image_fnames = ['anode', 'bimodal', 'nmc_wide', 'pre_kint', 'cracks']

# Mapping from contiguous labels returned by load_labels(ground_truths/*.png)
# (which sorts unique RGB colours by R + 255*G + 255^2*B) to the 0-indexed
# class predictions (where pred == k - 1 corresponds to user scribble label == k).
GT_TO_PRED_MAP: dict[str, dict[int, int]] = {
    'anode': {0: 2, 1: 1, 2: 0},
    'bimodal': {0: 2, 1: 1, 2: 0, 3: 3},
    'nmc_wide': {0: 2, 1: 1, 2: 0},
    'pre_kint': {0: 3, 1: 2, 2: 1, 3: 0},
    'cracks': {0: 2, 1: 1, 2: 0},
}


def remap_gt_labels(gt_raw: np.ndarray, mapping: dict[int, int]) -> np.ndarray:
    gt_mapped = np.zeros_like(gt_raw)
    for src_val, dst_val in mapping.items():
        gt_mapped[gt_raw == src_val] = dst_val
    return gt_mapped


images: list[Image.Image] = []
labels: list[np.ndarray] = []
ground_truths: list[np.ndarray] = []

for image_fname in image_fnames:
    img = Image.open(f"{ds_folder}/{image_fname}.png").convert('RGB')
    w, h = img.size
    ratio = w / h
    new_h = side_lengths[image_fname]
    new_w = int(ratio * new_h)
    img = img.resize((new_w, new_h))
    images.append(img)

    label = load_labels(f"{ds_folder}/{image_fname}_biased_labels.tiff")
    labels.append(label)

    gt_raw = load_labels(f"{ds_folder}/ground_truths/{image_fname}.png")
    gt_mapped = remap_gt_labels(gt_raw, GT_TO_PRED_MAP[image_fname])
    ground_truths.append(gt_mapped)

In [4]:
type AverageResult = tuple[np.ndarray, np.ndarray, float, float, np.ndarray]


def average_results(results: list[LinearProbeResult]) -> AverageResult:
    try:
        per_channel_r2 = np.array([res['per_channel_r_squared'] for res in results])
        mean_per_channel_r2 = np.mean(per_channel_r2, axis=0)
        std_per_channel_r2 = np.std(per_channel_r2, axis=0)
    except KeyError:
        mean_per_channel_r2 = np.zeros_like(0)
        std_per_channel_r2 = np.zeros_like(0)

    stack_r2 = np.array([res['stack_r_squared'] for res in results])
    mean_stack_r2 = float(np.mean(stack_r2))
    std_stack_r2 = float(np.std(stack_r2))

    mean_stack_pred = np.zeros_like(results[0]['stack_pred'])
    return (
        mean_per_channel_r2,
        std_per_channel_r2,
        mean_stack_r2,
        std_stack_r2,
        mean_stack_pred,
    )

In [5]:
ramp: RampTypes = 'lr+ud'
MASK_CUTOFF_FRAC = 1
STEP = 6
RANDOM_MASK = True
N_repeats = 6
CLASSICAL_PATCH_STRIDE = 14

preds_no_classical: dict[EncoderKey, list[np.ndarray]] = {k: [] for k in all_encoders}
preds_with_classical: dict[EncoderKey, list[np.ndarray]] = {k: [] for k in all_encoders}

if LOAD_RESULTS and os.path.exists(SCORES_CACHE_PATH):
    with open(SCORES_CACHE_PATH, "r") as f:
        cached_scores = json.load(f)
    per_image_r2: dict[EncoderKey, list[float]] = cached_scores["per_image_r2"]
    per_image_r2_std: dict[EncoderKey, list[float]] = cached_scores["per_image_r2_std"]
    micro_ds_r2: dict[EncoderKey, float] = cached_scores["micro_ds_r2"]
    micro_ds_r2_std: dict[EncoderKey, float] = cached_scores["micro_ds_r2_std"]
    micro_ds_r2['classical'] = -0.1
    mious_no_classical: dict[EncoderKey, list[float]] = cached_scores["mious"]["no_classical"]
    mious_with_classical: dict[EncoderKey, list[float]] = cached_scores["mious"]["with_classical"]
    print(f"Loaded cached R^2 and mIoU scores from {SCORES_CACHE_PATH}")
else:
    classical_features: list[np.ndarray] = [
        featurise(np.array(img.convert('L')), tr_cfg) for img in images
    ]

    per_image_r2 = {k: [] for k in all_encoders}
    per_image_r2_std = {k: [] for k in all_encoders}
    mious_no_classical = {k: [] for k in all_encoders}
    mious_with_classical = {k: [] for k in all_encoders}

    micro_ds_folder = 'data/linear_probe/homog_micros'
    micro_files = sorted(listdir(micro_ds_folder))
    micro_imgs = [Image.open(f"{micro_ds_folder}/{fname}").convert('RGB') for fname in micro_files]
    micro_classical_feats = [
        featurise(np.array(m_img.convert('L')), tr_cfg)[::CLASSICAL_PATCH_STRIDE, ::CLASSICAL_PATCH_STRIDE, :]
        for m_img in micro_imgs
    ]

    existing_micro_scores: dict[str, dict[str, float]] = {}
    for t_path in ("saved/t1_scores.json", "saved/t2_scores.json"):
        if os.path.exists(t_path):
            with open(t_path, "r") as f:
                existing_micro_scores.update(json.load(f).get("micro", {}))

    micro_ds_r2 = {}
    micro_ds_r2_std = {}

    # Load one model at a time and free VRAM after each encoder
    for key in all_encoders:
        model = (
            None
            if key == 'classical'
            else get_model_(key, DEVICE, True, "../../models/checkpoints", conf_dir="../../models/dinov3")
        )
        if model is not None:
            for blk in getattr(model.vit, 'blocks', []):
                if hasattr(blk, 'attn') and hasattr(blk.attn, 'fused_attn'):
                    blk.attn.fused_attn = True

        # 1. Per-image R^2 and segmentation mIoU (both ViT-only and ViT+classical) on target images
        for i, img in enumerate(images):
            classical = classical_features[i]
            image_arr = np.array(img)
            label = labels[i]
            gt = ground_truths[i]

            if key == 'classical':
                feats_hwc = classical[::CLASSICAL_PATCH_STRIDE, ::CLASSICAL_PATCH_STRIDE, :]
            else:
                with torch.no_grad():
                    feats = get_features(model, img, False)
                feats_hwc = np.transpose(feats, (1, 2, 0))

            probe_results: list[LinearProbeResult] = []
            for _ in range(N_repeats):
                res = do_linear_probe(
                    feats_hwc,
                    ramp,
                    probe_by_channel=False,
                    mask_step=STEP,
                    mask_cutoff_frac=MASK_CUTOFF_FRAC,
                    random_mask=RANDOM_MASK,
                    regressor='ridge',
                )
                probe_results.append(res)
            avg_res = average_results(probe_results)
            per_image_r2[key].append(avg_res[2])
            per_image_r2_std[key].append(avg_res[3])

            if key == 'classical':
                pred_cl, _, _ = train_and_apply(classical, label, tr_cfg, image=image_arr)
                miou_cl = float(class_avg_miou(pred_cl, gt))
                preds_no_classical[key].append(pred_cl)
                preds_with_classical[key].append(pred_cl)
                mious_no_classical[key].append(miou_cl)
                mious_with_classical[key].append(miou_cl)
                print(
                    f"{key} | {image_fnames[i]}: R^2 = {avg_res[2]:.3f} ± {avg_res[3]:.3f}, "
                    f"mIoU = {miou_cl:.3f}"
                )
            else:
                reduced = do_2D_pca(feats, 9, pre_norm='std', post_norm='minmax')
                reduced_hr = resize(
                    reduced,
                    (img.height, img.width),
                    order=1,
                    mode='reflect',
                    anti_aliasing=True,
                )
                combined_feats = concat_feats(reduced_hr, classical)

                pred_no_cl, _, _ = train_and_apply(reduced_hr, label, tr_cfg, image=image_arr)
                miou_no_cl = float(class_avg_miou(pred_no_cl, gt))
                preds_no_classical[key].append(pred_no_cl)
                mious_no_classical[key].append(miou_no_cl)

                pred_with_cl, _, _ = train_and_apply(combined_feats, label, tr_cfg, image=image_arr)
                miou_with_cl = float(class_avg_miou(pred_with_cl, gt))
                preds_with_classical[key].append(pred_with_cl)
                mious_with_classical[key].append(miou_with_cl)

                print(
                    f"{key} | {image_fnames[i]}: R^2 = {avg_res[2]:.3f} ± {avg_res[3]:.3f}, "
                    f"mIoU (ViT) = {miou_no_cl:.3f}, mIoU (ViT+classical) = {miou_with_cl:.3f}"
                )

        # 2. Microscopy dataset R^2 (data/linear_probe/homog_micros) as in T01_linear_probe.ipynb
        if key in existing_micro_scores:
            micro_ds_r2[key] = float(existing_micro_scores[key]["mean"])
            micro_ds_r2_std[key] = float(existing_micro_scores[key]["std"])
        else:
            micro_probe_results: list[LinearProbeResult] = []
            for m_idx, m_img in enumerate(micro_imgs):
                if key == 'classical':
                    m_feats_hwc = micro_classical_feats[m_idx]
                else:
                    with torch.no_grad():
                        m_feats_hwc = get_features(model, m_img, channel_last=True)
                for _ in range(N_repeats):
                    res = do_linear_probe(
                        m_feats_hwc,
                        ramp,
                        probe_by_channel=False,
                        mask_step=STEP,
                        mask_cutoff_frac=MASK_CUTOFF_FRAC,
                        random_mask=RANDOM_MASK,
                        regressor='ridge',
                    )
                    micro_probe_results.append(res)
            avg_micro = average_results(micro_probe_results)
            micro_ds_r2[key] = avg_micro[2]
            micro_ds_r2_std[key] = avg_micro[3]
        print(f"Microscopy DS R^2 | {key}: {micro_ds_r2[key]:.3f} ± {micro_ds_r2_std[key]:.3f}")

        del model
        torch.cuda.empty_cache()

    os.makedirs(os.path.dirname(SCORES_CACHE_PATH), exist_ok=True)
    with open(SCORES_CACHE_PATH, "w") as f:
        json.dump(
            {
                "image_fnames": image_fnames,
                "per_image_r2": per_image_r2,
                "per_image_r2_std": per_image_r2_std,
                "micro_ds_r2": micro_ds_r2,
                "micro_ds_r2_std": micro_ds_r2_std,
                "mious": {
                    "no_classical": mious_no_classical,
                    "with_classical": mious_with_classical,
                },
            },
            f,
            indent=2,
        )
    print(f"Saved R^2 and mIoU scores to {SCORES_CACHE_PATH}")

Loaded cached R^2 and mIoU scores from saved/s16_scores.json


In [6]:
image_titles = {
    'anode': 'Anode',
    'bimodal': 'Bimodal \ncathode',
    'nmc_wide': 'Cathode',
    'pre_kint': 'Cathode \n(+CC)',
    'cracks': 'Cathode \n(+cracks)',
}

preds_to_vis = preds_with_classical if ADD_CLASSICAL_FEATS else preds_no_classical
mious_to_vis = mious_with_classical if ADD_CLASSICAL_FEATS else mious_no_classical

# Only generate prediction visualisations if predictions were computed in this session
if any(len(v) > 0 for v in preds_to_vis.values()):
    plt.style.use("thesis.mplstyle")
    plt.rcParams['text.usetex'] = False
    add_custom_font('resources/fonts', 'Grotesk')

    pred_vis_dir = "saved/s16_preds"
    os.makedirs(pred_vis_dir, exist_ok=True)

    encoders_for_vis: list[EncoderKey] = (
        list(all_encoders) if INCLUDE_CLASSICAL_BASELINE else list(vit_models)
    )

    for x, fname in enumerate(image_fnames):
        overlay_arr = apply_labels_as_overlay(labels[x], images[x], COLORS, alpha=1)
        gt_arr = label2rgb(ground_truths[x] + 1, colors=COLORS[1:], bg_label=0)

        for key in encoders_for_vis:
            model_name = (
                "Classical"
                if key == 'classical'
                else MODEL_NAMES[key].replace('(COCO)', '').strip()
            )
            pred_arr = label2rgb(preds_to_vis[key][x] + 1, colors=COLORS[1:], bg_label=0)
            miou_val = mious_to_vis[key][x]
            r2_val = per_image_r2[key][x]

            fig_triplet, axs_t = plt.subplots(1, 3, figsize=(7.5, 2.3))
            axs_t[0].imshow(overlay_arr, aspect='auto', interpolation='nearest')
            axs_t[0].set_title(f"{image_titles[fname]}: Image + labels")
            hide_axis(axs_t[0])

            axs_t[1].imshow(gt_arr, aspect='auto', interpolation='nearest')
            axs_t[1].set_title("Ground truth")
            hide_axis(axs_t[1])

            axs_t[2].imshow(pred_arr, aspect='auto', interpolation='nearest')
            axs_t[2].set_title(f"{model_name} (mIoU={miou_val:.2f}, $R^2$={r2_val:.2f})")
            hide_axis(axs_t[2])

            plt.tight_layout()
            fig_triplet.savefig(
                f"{pred_vis_dir}/{fname}_{key}.png", dpi=200, bbox_inches="tight"
            )
            plt.close(fig_triplet)

    N_ROWS = len(encoders_for_vis) + 2
    N_COLS = len(image_fnames)
    fig, axs = plt.subplots(N_ROWS, N_COLS, figsize=(7.5, 2.3 * 4.5))

    for y in range(N_ROWS):
        for x, fname in enumerate(image_fnames):
            if y == 0:
                if x == 0:
                    axs[y, 0].set_ylabel("Image + labels")
                axs[y, x].set_title(image_titles[fname])
                arr = apply_labels_as_overlay(labels[x], images[x], COLORS, alpha=1)
            elif y == 1:
                if x == 0:
                    axs[y, 0].set_ylabel("Ground truth")
                arr = label2rgb(ground_truths[x] + 1, colors=COLORS[1:], bg_label=0)
            else:
                key = encoders_for_vis[y - 2]
                if x == 0:
                    model_name = (
                        "Classical"
                        if key == 'classical'
                        else MODEL_NAMES[key].replace('(COCO)', '').strip()
                    )
                    weight = 700 if 'alibi' in str(key).lower() else 500
                    axs[y, 0].set_ylabel(model_name, weight=weight)
                arr = preds_to_vis[key][x]
                arr = label2rgb(arr + 1, colors=COLORS[1:], bg_label=0)
            axs[y, x].imshow(arr, aspect='auto', interpolation='nearest')
            hide_axis(axs[y, x])

    plt.tight_layout()
    fig.savefig(f"{pred_vis_dir}/all_predictions_grid.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{pred_vis_dir}/all_predictions_grid.pdf", dpi=300, bbox_inches="tight")
    plt.close(fig)

In [7]:
plt.style.use("thesis.mplstyle")
plt.rcParams['text.usetex'] = False
add_custom_font('resources/fonts', 'Grotesk')

model_colors: dict[EncoderKey, str] = {
    'dino_s': '#648FFF',
    'dinov2_s': '#785EF0',
    'alibi_coco_dinov2_s': '#16ce37',
    'sam_b': '#DC267F',
    'clip_b': '#FE6100',
    'mae_b': '#FFB000',
    'in1k_b': '#009E73',
    'classical': '#000000',
}

model_display_names: dict[EncoderKey, str] = {
    'dino_s': 'DINO',
    'dinov2_s': 'DINOv2',
    'alibi_coco_dinov2_s': 'ALiBi-DINOv2',
    'sam_b': 'SAM',
    'clip_b': 'CLIP',
    'mae_b': 'MAE',
    'in1k_b': 'ViT (IN1k)',
    'classical': 'Classical',
}

image_markers: dict[str, str] = {
    'anode': 'o',
    'bimodal': 's',
    'nmc_wide': '^',
    'pre_kint': 'D',
    'cracks': 'v',
}

base_encoder_order: tuple[ModelTypes, ...] = (
    'alibi_coco_dinov2_s',
    'in1k_b',
    'clip_b',
    'sam_b',
    'dino_s',
    'dinov2_s',
    'mae_b',
)
legend_encoder_order: tuple[EncoderKey, ...] = (
    (*base_encoder_order, 'classical')
    if INCLUDE_CLASSICAL_BASELINE
    else base_encoder_order
)


def get_x_r2(key: EncoderKey, img_idx: int) -> float:
    if USE_MICRO_DS_R2:
        return float(micro_ds_r2[key])
    return float(per_image_r2[key][img_idx])


def plot_panel(ax: plt.Axes, mious_dict: dict[EncoderKey, list[float]], title: str | None = None) -> None:
    for key in legend_encoder_order:
        for i, fname in enumerate(image_fnames):
            ax.scatter(
                get_x_r2(key, i),
                mious_dict[key][i],
                color=model_colors[key],
                marker=image_markers[fname],
                s=36,
                alpha=0.9,
                edgecolors='black',
                linewidths=0.4,
            )
    if title is not None:
        ax.set_title(title, weight=500)
    ax.set_ylabel("Segmentation mIoU")
    ax.set_xlim(right=1.05)
    ax.grid(alpha=0.2)


if PLOT_BOTH_CLASSICAL_MODES:
    fig, axs = plt.subplots(2, 1, figsize=(4.25, 2 * 2.3), sharex=True, sharey=True)
    plot_panel(axs[0], mious_no_classical, title="ViT")
    plot_panel(axs[1], mious_with_classical, title="ViT (+classical)")
    axs[1].set_xlabel(r"Positional $R^2$ (lr+ud)")
    legend_ax = axs[0]
    leg2_anchor = (1.02, -0.25)
else:
    fig, ax = plt.subplots(figsize=(4, 2 * 2.3))
    active_mious = mious_with_classical if ADD_CLASSICAL_FEATS else mious_no_classical
    plot_panel(ax, active_mious, title=None)
    ax.set_xlabel(r"Positional $R^2$ (lr+ud)")
    legend_ax = ax
    leg2_anchor = (1.02, 0.0)

encoder_handles = [
    Line2D(
        [0],
        [0],
        marker='o',
        color='w',
        label=model_display_names[k],
        markerfacecolor=model_colors[k],
        markeredgecolor='black',
        markeredgewidth=0.4,
        markersize=6,
    )
    for k in legend_encoder_order
]

image_handles = [
    Line2D(
        [0],
        [0],
        marker=image_markers[fname],
        color='w',
        label=image_titles[fname],
        markerfacecolor='#666666',
        markeredgecolor='black',
        markeredgewidth=0.4,
        markersize=6,
    )
    for fname in image_fnames
]

legend_box_width = 0.28
leg1 = legend_ax.legend(
    handles=encoder_handles,
    title="Encoder",
    loc="upper left",
    bbox_to_anchor=(1.02, 1.0, legend_box_width, 0.0),
    mode="expand",
    frameon=True,
)
legend_ax.add_artist(leg1)
leg2 = legend_ax.legend(
    handles=image_handles,
    title="Image",
    loc="lower left" if not PLOT_BOTH_CLASSICAL_MODES else "upper left",
    bbox_to_anchor=(*leg2_anchor, legend_box_width, 0.0),
    mode="expand",
    frameon=True,
)

# Horizontally centre "Encoder" and "Image" titles within their equal-width legend boxes
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
for leg in (leg1, leg2):
    frame_bbox = leg.get_frame().get_window_extent(renderer)
    title_bbox = leg.get_title().get_window_extent(renderer)
    dx_pts = ((0.5 * (frame_bbox.x0 + frame_bbox.x1)) - (0.5 * (title_bbox.x0 + title_bbox.x1))) * 72.0 / fig.dpi
    leg.get_title().set_position((dx_pts, 0))

SAVE = True
if SAVE:
    if PLOT_BOTH_CLASSICAL_MODES:
        name = "saved/S16_both.pdf"
    else:
        name = "saved/S16.pdf" if ADD_CLASSICAL_FEATS else "saved/S16_no_classical.pdf"
    plt.savefig(name, dpi=300, bbox_inches="tight")
    plt.close()
else:
    plt.show()

findfont: Failed to find font weight normal for Host Grotesk, now using 300.
findfont: Failed to find font weight normal for Host Grotesk, now using 300.
findfont: Failed to find font weight 500 for Host Grotesk, now using 300.
findfont: Failed to find font weight normal for Host Grotesk, now using 300.
findfont: Failed to find font weight normal for Host Grotesk, now using 300.
